In [2]:

!pip install --quiet numpy pandas scikit-learn requests yfinance holidays pytrends pytorch_forecasting lightning optuna lion-pytorch optuna-integration[pytorch_lightning]

# Posodobitev Optune (potrebno za pravilni uvoz lightning callbacka)
!pip install --quiet --upgrade optuna

# Osnovni importi
import os
import sys
import copy
import numpy as np
import pandas as pd
import torch
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from pylab import rcParams
import yfinance as yf

import pickle
from copy import deepcopy as dc

# PyTorch nastavitve
torch.set_float32_matmul_precision('high')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer

# Lightning & forecasting
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.tuner import Tuner
from lightning.pytorch.callbacks import Callback

from pytorch_forecasting import (
    TimeSeriesDataSet,
    GroupNormalizer,
    NaNLabelEncoder,
    TemporalFusionTransformer
)
from pytorch_forecasting.data import MultiNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

# Lion optimizer
from lion_pytorch import Lion

# Optuna & patch za `is_overridden` bug
import optuna
from optuna.integration import PyTorchLightningPruningCallback

import pytorch_lightning.utilities.model_helpers as mh


def patched_is_overridden(method_name, instance, parent=None):
    if parent is None:
        parent = Callback
    from lightning_utilities.core.overrides import is_overridden as _is_overridden
    return _is_overridden(method_name, instance, parent)

mh.is_overridden = patched_is_overridden


# Nastavitve grafov in opozoril
rcParams["figure.figsize"] = (12, 5)
warnings.filterwarnings("ignore")


zsh:1: no matches found: optuna-integration[pytorch_lightning]
Using device: cpu


In [3]:
df = pd.read_csv("Data/daily_data_cleaned.csv", parse_dates=['Date'])
df["Close"] = np.log(df["Close"])

cat_cols = ['US', 'UK', 'Japan', 'China', 'day', 'month', 'day_of_week', 'week_of_year', 'year', 'quarter',
            'is_weekend', 'is_month_end', 'time_idx', 'Halving', "group"]

df[cat_cols] = df[cat_cols].astype(str).astype("category")

df["time_idx"] = df["time_idx"].astype(int)

In [4]:
# === Known Reals ===
time_varying_known_reals = [
       'GDP and National Income', 'Personal Income and Employment',
       'Industry Specific Accounts', 'Fixed Assets and Investment',
       'Trade and International Transactions', 'Government and Public Sector',
       'Financial and Corporate Data',
       "time_idx"
]

# === Unknown Reals ===
time_varying_unknown_reals = [
    "High", "Low", "Open", "Volume", "BTC_miners",
    "BTC_transactions", "BTC_network", "Unique Addresses Used",
    "Number of Transactions", "Transactions Per Second", "Output Volume",
    "Mempool Transaction Count", "Mempool Size Growth", "Mempool Size (Bytes)",
    "Transactions Excluding Popular Addresses", "Estimated Transaction Volume (BTC)",
    "Estimated Transaction Volume (USD)", "Miners Revenue (USD)",
    "Transaction Fees (BTC)", "Transaction Fees (USD)",
    "Cost per Transaction (%)", "Cost per Transaction (USD)",
    "Network Difficulty", "Hash Rate (TH/s)", "Block Size",
    "Average Block Size", "Transactions per Block", "Trade Volume",
    "Total Bitcoins", "Market Cap", "M2SL", "M1SL", "WALCL", "CPIAUCSL",
    "CPILFESL", "CUSR0000SA0L2", "FEDFUNDS", "IRLTLT01JPM156N", "PAYEMS",
    "UNRATE", "CIVPART", "DSPIC96", "PCE", "PSAVERT", "GEPUCURRENT",
    "USEPUINDXD", "EPUMONETARY", "APU000072610", "CPIENGSL", "HOUST",
    "PERMIT", "RECPROUSM156N", "DTWEXBGS", "DGS10", "DGS2", "DGS30",
    "T10Y2Y", "T10Y3M", "T10YIE", "DFF", "SP500", "VIXCLS", "NASDAQCOM",
    "BAA10Y", "DCOILWTICO", "WLEMUINDXD", "T5YIFR", "Close"
]


# === Known Categoricals ===
time_varying_known_categoricals = [
       'US', 'UK', 'Japan','China', 'day', 'month', 'day_of_week', 'week_of_year', 'year',
       'quarter', 'is_weekend', 'is_month_end', 'Halving'
]

static_categoricals = ["group"]



In [5]:
class MyTFT(TemporalFusionTransformer):
    def __init__(self, *args, beta1=0.9, beta2=0.999, weight_decay=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.save_hyperparameters("learning_rate")
        self.hparams.beta1 = beta1
        self.hparams.beta2 = beta2
        self.hparams.weight_decay = weight_decay

    def configure_optimizers(self):
        optimizer = Lion(
            self.parameters(),
            lr=self.hparams.learning_rate,
            betas=(self.hparams.beta1, self.hparams.beta2),
            weight_decay=self.hparams.weight_decay
        )
        scheduler = {
            "scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=3,
                min_lr=1e-5,
                verbose=True
            ),
            "monitor": "val_loss",
            "interval": "epoch",
            "frequency": 1
        }
        return {"optimizer": optimizer, "lr_scheduler": scheduler}


    def on_train_epoch_end(self):
        current_lr = self.trainer.optimizers[0].param_groups[0]['lr']
        self.log("lr", current_lr, prog_bar=True, logger=True)

In [7]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor, Callback
from lightning.pytorch.loggers import TensorBoardLogger
import lightning.pytorch as pl
import os


# === Parametri ===
params = {'learning_rate': 0.00329, 'hidden_size': 128, 'hidden_continuous_size': 64, 'attention_head_size': 4, 'dropout': 0, 'gradient_clip_val': 0.100000000000001, 'weight_decay': 1e-15, 'beta1': 0.918, 'beta2': 0.992, 'max_encoder_length': 64,  "batch_size":16}


max_prediction_length = 31
max_encoder_length = params["max_encoder_length"]
training_cutoff = df["time_idx"].max() - max_prediction_length

categorical_encoders = {col: NaNLabelEncoder(add_nan=True) for col in time_varying_known_categoricals}
pl.seed_everything(42, workers=True)

target_normalizer = None #GroupNormalizer(groups=["group"],transformation="log", method="robust")

# === Dataset ===
training = TimeSeriesDataSet(
    df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="Close",
    group_ids=["group"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_categoricals,
    time_varying_known_categoricals=time_varying_known_categoricals,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=target_normalizer,
    add_relative_time_idx=True,
    add_target_scales=False,
    add_encoder_length=True,
    allow_missing_timesteps=True,
    categorical_encoders=categorical_encoders,
)


validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

train_dataloader = training.to_dataloader(train=True, batch_size=params["batch_size"], num_workers=7)
val_dataloader = validation.to_dataloader(train=False, batch_size=params["batch_size"] * 10, num_workers=7)


# === Model ===
model = MyTFT.from_dataset(
    training,
    learning_rate=params["learning_rate"],
    hidden_size=params["hidden_size"],
    attention_head_size=params["attention_head_size"],
    dropout=params["dropout"],
    hidden_continuous_size=params["hidden_continuous_size"],
    loss=QuantileLoss([0.1, 0.5, 0.9]),
    log_interval=1,
    reduce_on_plateau_patience=4,
    beta1=params["beta1"],
    beta2=params["beta2"],
    weight_decay=params["weight_decay"],
)

# === Logger & Callbacks ===
log_dir = "lightning_logs"
logger = TensorBoardLogger(save_dir=log_dir, name="tft_optuna_final")
callbacks = [
    LearningRateMonitor(logging_interval="epoch"),
    EarlyStopping(monitor="val_loss", min_delta=.01, patience=8, mode="min", verbose=True),
    ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")
]

# === Trainer ===
trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    precision=32,
    accumulate_grad_batches=2,
    gradient_clip_val=params["gradient_clip_val"],
    callbacks=callbacks,
    enable_progress_bar=True,
    enable_model_summary=True,
    deterministic=False,
    logger=logger,
    check_val_every_n_epoch=1,
    num_sanity_val_steps=2,
    benchmark=True,
)

trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)



Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 1.8 K  | train
3  | prescalers                         | ModuleDict                      | 10.0 K | train
4  | static_variable_selection          | VariableSelectionNetwork        | 26.3 K | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 2.5 M  | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 255 K  | train
7  | static_context_variable_selection  | Ga

Epoch 0: 100%|██████████| 31/31 [00:25<00:00,  1.20it/s, v_num=0, train_loss_step=0.941]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 31/31 [00:34<00:00,  0.89it/s, v_num=0, train_loss_step=0.941, val_loss=0.804, train_loss_epoch=4.870]

Metric val_loss improved. New best score: 0.804


Epoch 1: 100%|██████████| 31/31 [00:31<00:00,  0.98it/s, v_num=0, train_loss_step=0.184, val_loss=0.804, train_loss_epoch=4.870, lr=0.00329]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 31/31 [00:40<00:00,  0.76it/s, v_num=0, train_loss_step=0.184, val_loss=0.301, train_loss_epoch=0.338, lr=0.00329]

Metric val_loss improved by 0.503 >= min_delta = 0.01. New best score: 0.301


...


Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 11:   0%|          | 0/31 [00:00<?, ?it/s, v_num=0, train_loss_step=0.0812, val_loss=0.0669, train_loss_epoch=0.0869, lr=0.00164]         

python(22224) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22226) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22227) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22228) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22229) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22230) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(22231) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Epoch 11:  42%|████▏     | 13/31 [00:24<00:33,  0.53it/s, v_num=0, train_loss_step=0.0639, val_loss=0.0669, train_loss_epoch=0.0869, lr=0.00164]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:

# === Shrani model ===
os.makedirs(path  + "MODELI", exist_ok=True)
model_path = os.path.join(path + "MODELI", "TFT_FINAL_1.ckpt")
trainer.save_checkpoint(model_path)